# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neha-raniii/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/neha-raniii/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd
import numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} rows loaded")

30,000 rows loaded


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Looking at the distributions of key fields before testing any signal - checking for heavy tails that could make a naive average misleading.

In [2]:
key_fields = ['impressions_90d', 'avg_position', 'ctr', 'days_since_last_update']
print(df[key_fields].describe().round(2))


       impressions_90d  avg_position       ctr  days_since_last_update
count         30000.00      30000.00  30000.00                30000.00
mean           5200.37         16.34      0.51                   46.10
std           16838.02         15.22      3.28                   42.08
min               1.00          0.00      0.00                    1.00
25%              81.00          6.20      0.00                   20.00
50%             731.00         10.80      0.07                   20.00
75%            3615.25         22.30      0.29                  104.00
max          517715.00        245.00    100.00                  373.00


Heavy tails observed: impressions_90d has a mean (5,200) far above its median (731) - a small number of very high-traffic pages (max 517,715) pull the average up. Similarly, ctr's mean (0.51) sits well above its median (0.07), with a max of 100 - a few extreme outliers. This means any signal test using ctr or impressions_90d should compare medians or grouped rates, not raw averages, or a handful of outlier pages will dominate the read.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Three signal tests with verdicts: (1) staleness vs decline - rechecking a Week-4 finding, (2) CTR vs position tier - rechecking a Week-4 confirmed signal, (3) word count vs decline - a new test.

In [3]:
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

# Test 1: staleness
df['is_stale'] = df['days_since_last_update'] >= 180
t1 = df.groupby('is_stale')['is_declining'].mean().round(3)
print("Test 1 - Staleness vs decline rate:")
print(t1)

# Test 2: CTR vs position tier
t2 = df.groupby('position_tier')['ctr'].mean().sort_values(ascending=False).round(3)
print("\nTest 2 - Mean CTR by position tier:")
print(t2)

# Test 3: word count vs decline
df['word_count_filled'] = df['word_count'].fillna(0)
df['word_bucket'] = pd.cut(df['word_count_filled'], bins=[0, 1000, 2000, 3000, 100000], labels=['<1K','1K-2K','2K-3K','3K+'])
t3 = df.groupby('word_bucket', observed=True)['is_declining'].mean().round(3)
print("\nTest 3 - Decline rate by word count bucket:")
print(t3)

Test 1 - Staleness vs decline rate:
is_stale
False    0.542
True     0.471
Name: is_declining, dtype: float64

Test 2 - Mean CTR by position tier:
position_tier
top_3       1.484
page_1      0.652
striking    0.323
page_3_5    0.222
deep        0.150
Name: ctr, dtype: float64

Test 3 - Decline rate by word count bucket:
word_bucket
<1K      0.207
1K-2K    0.555
2K-3K    0.587
3K+      0.595
Name: is_declining, dtype: float64


Verdicts:

Test 1 - Staleness vs decline: OPPOSITE. Stale pages (180+ days since update) show a LOWER decline rate (47.1%) than fresh pages (54.2%) - the same finding as Week 4, now reconfirmed. Staleness alone does not predict decline the way the intuition behind refresh flags assumes.

Test 2 - CTR vs position tier: CONFIRMED. CTR drops cleanly by tier: top_3 (1.484) > page_1 (0.652) > striking (0.323) > page_3_5 (0.222) > deep (0.150). This is a clean, monotonic, trustworthy signal.

Test 3 - Word count vs decline: OPPOSITE (of the common belief that longer content is safer/better). Decline rate actually rises with word count: <1K words decline only 20.7% of the time, while 3K+ words decline 59.5% of the time. This does not mean longer content causes decline - it's more likely that longer, more competitive pages face harder competition or slower content decay cycles. This is an association, not a causal claim.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

Flag-linked test: FlyRank's own "stale_visible_page" flag logic assumes days_since_last_update >= 180 AND impressions_90d >= 500 signals a page worth reviewing for decline. Testing whether the "stale" half of that assumption actually correlates with decline on this data (already computed in Test 1 above).

In [4]:
n_stale = df['is_stale'].sum()
n_total = len(df)
print(f"Pages meeting the stale threshold (180+ days): {n_stale:,} of {n_total:,} ({n_stale/n_total*100:.1f}%)")
print(f"\nDecline rate - stale pages: {df[df['is_stale']]['is_declining'].mean():.3f}")
print(f"Decline rate - fresh pages: {df[~df['is_stale']]['is_declining'].mean():.3f}")
print(f"\nVerdict: staleness alone does NOT support the flag's underlying assumption on this dataset.")


Pages meeting the stale threshold (180+ days): 174 of 30,000 (0.6%)

Decline rate - stale pages: 0.471
Decline rate - fresh pages: 0.542

Verdict: staleness alone does NOT support the flag's underlying assumption on this dataset.


This flag-linked test confirms, with an explicit denominator, what Test 1 showed: only 174 of 30,000 pages (0.6%) meet the "stale" threshold used by FlyRank's stale_visible_page flag logic. Among those 174, the decline rate (47.1%) is actually LOWER than among the 29,826 fresh pages (54.2%). The flag's underlying assumption - that staleness alone signals decline risk - is not supported on this dataset. This does not mean the flag is useless (it may still catch worthwhile review candidates for other reasons), but staleness by itself should not be treated as a reliable decline predictor without the impressions_90d >= 500 co-condition and probably additional signals like the CTR-position pattern that DID hold up.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [5]:
"""What this means in practice: A content team relying purely on "how long since this page was updated" to prioritize review work would be prioritizing the wrong pages more often than not - the CTR-vs-position gap is a stronger, cleaner signal on this data. Word count should also not be used as a simple safety signal; longer pages decline more often here, likely reflecting harder competition rather than content quality. The practical takeaway: build review priority from CTR-position gaps and demand (impressions), and treat staleness as one input a reviewer checks manually - not a standalone flag to trust automatically."""

'What this means in practice: A content team relying purely on "how long since this page was updated" to prioritize review work would be prioritizing the wrong pages more often than not - the CTR-vs-position gap is a stronger, cleaner signal on this data. Word count should also not be used as a simple safety signal; longer pages decline more often here, likely reflecting harder competition rather than content quality. The practical takeaway: build review priority from CTR-position gaps and demand (impressions), and treat staleness as one input a reviewer checks manually - not a standalone flag to trust automatically.'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.